# D-Residue Error Validation Against Experimental PDB Data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tommaso-R-Marena/ChiralFold/blob/master/demos/D_Residue_Experimental_Validation.ipynb)

**Reproducible in silico validation** of the 16 PDB structures with D-label/L-coordinate mismatches.

This notebook addresses reviewer concerns about single-author error classification by cross-checking each case against:
- RCSB deposition metadata (X-ray / NMR method, resolution, title)
- Independent signed Cα tetrahedron volume (numpy only)
- wwPDB Chemical Component Dictionary InChI stereochemistry for CCD-code misassignments

Also documents **PDB 5M2K** (vancomycin–Zn²⁺ glycopeptide) as a non-protein Ramachandran benchmark exclusion.

**Runtime:** ~2 minutes on Colab CPU. No GPU required.

In [ ]:
# Clone ChiralFold and install minimal dependencies
!git clone --depth 1 https://github.com/Tommaso-R-Marena/ChiralFold.git
%cd ChiralFold
!pip install -q numpy scipy pandas matplotlib

In [ ]:
# Run the independent validation pipeline (queries RCSB REST API)
!python benchmarks/experimental_structure_validation.py

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown

with open('results/experimental_validation_report.json') as f:
    report = json.load(f)

df = pd.read_csv('results/experimental_validation_summary.csv')
display(Markdown(f"""
### Validation summary
- **Structures validated:** {report['n_structures']}
- **Automated pass (objective criteria):** {report['n_automated_pass']}
- **Borderline (manual review):** {report['n_borderline']}
- **Experimental methods:** {', '.join(report['methods'])}
"""))
display(df)

In [ ]:
# 5M2K: confirm non-protein benchmark exclusion (vancomycin glycopeptide, not a globular protein)
with open('results/5m2k_benchmark_exclusion.json') as f:
    m2k = json.load(f)

display(Markdown(f"""
### PDB 5M2K benchmark exclusion
| Field | Value |
|-------|-------|
| Title | {m2k['title']} |
| Molecule | {m2k['molecule']} |
| Method | {m2k['experimental_method']} @ {m2k['resolution_angstrom']} Å |
| Sequence | `{m2k['entity_sequence']}` |
| Non-standard monomers | {m2k['non_standard_monomers']} / 7 |
| Protein benchmark entry? | **{m2k['is_protein_benchmark_entry']}** |

{m2k['benchmark_exclusion_reason']}
"""))

In [ ]:
# Per-structure detail: 1HHZ (0.99 Å X-ray — strongest stereochemistry error case)
hh = next(s for s in report['structures'] if s['pdb_id'] == '1HHZ')
display(Markdown(f"""
### Example: 1HHZ (ultra-high-resolution X-ray)
- **Title:** {hh['deposition_title']}
- **Resolution:** {hh['resolution_angstrom']} Å
- **Signed volume:** {hh['checks']['signed_volume_range']} Å³ (L-coordinates at D-label)
- **Automated validation:** {hh['automated_validation_pass']}
- **Rationale:** {hh['validation_rationale']}
"""))

## Optional: mmCIF-native re-verification

The primary survey used legacy PDB files; **245 mmCIF-only entries** were excluded from per-code coverage statistics. Re-verify known error structures directly from mmCIF:

```bash
pip install gemmi
python benchmarks/mmcif_d_residue_expansion.py
```

On **Rockfish** (UMD HPC), use an interactive CPU node or submit:
```bash
module load anaconda
srun --partition=shared --ntasks=1 --cpus-per-task=4 --mem=8G --time=02:00:00 bash
git clone https://github.com/Tommaso-R-Marena/ChiralFold.git && cd ChiralFold
pip install --user numpy gemmi
python benchmarks/experimental_structure_validation.py
```

## Lean 4 proof generalization (Harmonic Aristotle)

To address formal-proof scope concerns, paste `formal/chirality_nogo/ARISTOTLE_PROMPT.md` into [Harmonic Aristotle](https://aristotle.harmonic.fun/) with the Lean project from branch `cursor/aristotle-formal-proofs-9901`.